In [1]:
import torch
from torch import nn
from torch.func import vmap, grad, jacrev, jacfwd
import torch.utils.benchmark as benchmark

In [2]:
# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Fetching the device that will be used throughout this notebook
device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda:0")
print("Using device", device)

Using device cuda:0


In [3]:
def get_activation_derivative(activation):
    if isinstance(activation, nn.Tanh):
        return lambda x: 1 - torch.tanh(x)**2
    elif isinstance(activation, nn.ELU):
        return lambda x: (x<=0) * activation.alpha * torch.exp(x) + (x>0) * 1.
    elif isinstance(activation, nn.ReLU):
        return lambda x: (x<=0) * 0. + (x>0) * 1.
    elif isinstance(activation, nn.LeakyReLU):
        return lambda x: (x<=0) * activation.negative_slope + (x>0) * 1.
    elif isinstance(activation, nn.Sigmoid):
        return lambda x: torch.sigmoid(x) - torch.sigmoid(x)**2
    else:
        return None
    
def dydx(x, mlp):
    """Compute jacobian of a feedforward network with one hidden layer.
    
    y = W2 * a
    a = activation(z)
    z = W1 * x + b1 
    
    -> dydx = dyda * dadz * dzdx 
            = W2 * dadz * W1

    args: 
        x: inputs of shape (bs, n_in)
        mlp: a single hidden layer network 
    """
    layer_1, act, layer_2 = mlp[0], mlp[1], mlp[2]
    z = layer_1(x)  # (bs, n_hidden)

    act_deriv = get_activation_derivative(act)
    if act_deriv is not None:
        dadz = act_deriv(z)  # (bs, n_hidden)
        dadz = torch.diag_embed(dadz)  # (bs, n_hidden, n_hidden)
    else:  # compute using autodiff, about 10 times slower than explicit computation
        dadz = vmap(jacrev(act))(z)

    res = torch.matmul(layer_2.weight, dadz)  # (n_out, n_hidden) @ (bs, n_hidden, n_hidden) -> (bs, n_out, n_hidden)
    res = torch.matmul(res, layer_1.weight)  # (bs, n_out, n_hidden) @ (n_hidden, n_in) -> (bs, n_out, n_in)
    return res  # (bs, n_out, n_in)

In [4]:
def func1(x, mlp):
    return vmap(grad(lambda p: mlp(p).squeeze()))(x)  # (bs, n_in)

def func2(x, mlp):
    return vmap(jacrev(mlp))(x)  # (bs, n_out, n_in)

def func3(x, mlp):
    jac = jacrev(mlp)(x)  # (bs, n_out, bs, n_in)
    jac = torch.diagonal(jac, dim1=0, dim2=2)  # (n_out, n_in, bs)
    jac = torch.permute(jac, (2, 0, 1))  # (bs, n_out, n_in)
    return jac

def func4(x, mlp):
    return vmap(jacfwd(mlp))(x)  # (bs, n_out, n_in)

def func5(x, mlp):
    jac = jacfwd(mlp)(x)  # (bs, n_out, bs, n_in)
    jac = torch.diagonal(jac, dim1=0, dim2=2)  # (n_out, n_in, bs)
    jac = torch.permute(jac, (2, 0, 1))  # (bs, n_out, n_in)
    return jac

In [5]:
mlp = nn.Sequential(
    nn.Linear(2, 10), 
    nn.Sigmoid(), 
    nn.Linear(10, 1)
)
mlp.to(device)

x = torch.randn(100, 2, device=device, requires_grad=False)

In [6]:
jac = dydx(x, mlp)
jac1 = func1(x, mlp)
jac2 = func2(x, mlp)
jac3 = func3(x, mlp)
jac4 = func4(x, mlp)
jac5 = func5(x, mlp)

print(jac.shape)
print(jac1.shape)
print(jac2.shape)
print(jac3.shape)
print(jac4.shape)
print(jac5.shape)

print(torch.linalg.norm(jac2 - jac))
assert torch.allclose(jac2, jac)

print(torch.linalg.norm(jac2.squeeze(-2) - jac1))
assert torch.allclose(jac2.squeeze(-2), jac1)

print(torch.linalg.norm(jac2 - jac3))
assert torch.allclose(jac2, jac3)

print(torch.linalg.norm(jac2 - jac4))
assert torch.allclose(jac2, jac4)

print(torch.linalg.norm(jac2 - jac5))
assert torch.allclose(jac2, jac5)

print(torch.linalg.norm(jac4 - jac5))
assert torch.allclose(jac4, jac5)


torch.Size([100, 1, 2])
torch.Size([100, 2])
torch.Size([100, 1, 2])
torch.Size([100, 1, 2])
torch.Size([100, 1, 2])
torch.Size([100, 1, 2])
tensor(6.2318e-08, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
tensor(0., device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
tensor(0., device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
tensor(6.8442e-08, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
tensor(6.3174e-08, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
tensor(6.8839e-08, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)


In [7]:
t_forward = benchmark.Timer(
    stmt="mlp(x)",
    globals={"x": x, "mlp": mlp})

t0 = benchmark.Timer(
    stmt="dydx(x, mlp)",
    setup="from __main__ import dydx",
    globals={"x": x, "mlp": mlp})

t1 = benchmark.Timer(
    stmt="func1(x, mlp)",
    setup="from __main__ import func1",
    globals={"x": x, "mlp": mlp})

t2 = benchmark.Timer(
    stmt="func2(x, mlp)",
    setup="from __main__ import func2",
    globals={"x": x, "mlp": mlp})

t3 = benchmark.Timer(
    stmt="func3(x, mlp)",
    setup="from __main__ import func3",
    globals={"x": x, "mlp": mlp})

t4 = benchmark.Timer(
    stmt="func4(x, mlp)",
    setup="from __main__ import func4",
    globals={"x": x, "mlp": mlp})

t5 = benchmark.Timer(
    stmt="func5(x, mlp)",
    setup="from __main__ import func5",
    globals={"x": x, "mlp": mlp})


print(t_forward.timeit(1000))
print(t0.timeit(1000))
print(t1.timeit(1000))
print(t2.timeit(1000))
print(t3.timeit(1000))
print(t4.timeit(1000))
print(t5.timeit(1000))


mlp(x)
  66.12 us
  1 measurement, 1000 runs , 1 thread
dydx(x, mlp)
setup: from __main__ import dydx
  176.60 us
  1 measurement, 1000 runs , 1 thread
func1(x, mlp)
setup: from __main__ import func1
  1.02 ms
  1 measurement, 1000 runs , 1 thread
func2(x, mlp)
setup: from __main__ import func2
  1.46 ms
  1 measurement, 1000 runs , 1 thread
func3(x, mlp)
setup: from __main__ import func3
  789.42 us
  1 measurement, 1000 runs , 1 thread
func4(x, mlp)
setup: from __main__ import func4
  1.48 ms
  1 measurement, 1000 runs , 1 thread
func5(x, mlp)
setup: from __main__ import func5
  1.21 ms
  1 measurement, 1000 runs , 1 thread


In [8]:
jac = dydx(x, mlp)
loss = torch.nn.functional.mse_loss(jac.squeeze(-2), x)
loss.backward()

print(mlp[0].weight.grad)
print(mlp[0].bias.grad)
print(mlp[2].weight.grad)
print(mlp[2].bias.grad)


tensor([[ 7.4906e-04,  2.2665e-05],
        [-4.7539e-03, -2.6285e-03],
        [ 7.7656e-04,  2.9834e-03],
        [-9.3151e-04,  1.3511e-03],
        [ 3.3876e-04, -3.8056e-03],
        [ 1.9573e-03, -1.6821e-03],
        [-1.9291e-03,  2.3506e-03],
        [-1.5829e-03, -3.3724e-03],
        [ 5.4982e-03, -9.6331e-03],
        [-8.1646e-04,  7.8486e-04]], device='cuda:0')
tensor([-0.0009, -0.0026,  0.0104,  0.0015, -0.0022, -0.0011,  0.0091, -0.0050,
        -0.0056,  0.0012], device='cuda:0')
tensor([[-0.0012,  0.0049, -0.0127,  0.0196,  0.0015, -0.0172, -0.0050, -0.0215,
         -0.0242,  0.0083]], device='cuda:0')
None


In [28]:
p = torch.randn(100, 2, device=device, requires_grad=False)
q = torch.randn(100, 2, device=device, requires_grad=False)

shift = nn.Parameter(torch.zeros(2), requires_grad=True).to(device)

def f(p, q):
    grad_V = dydx(p, mlp)
    grad_V = grad_V.squeeze(dim=-1)
    p_new = -q + grad_V
    q_new = p + shift 
    return 

t0 = benchmark.Timer(
    stmt="""
    f(p, q)
    """,
    setup="from __main__ import f",
    globals={"p": p, "q": q})

t0.timeit(100)

f(p, q)
setup: from __main__ import f
  244.68 us
  1 measurement, 100 runs , 1 thread